# A fake orders API

In [ ]:
import os, random
from pathlib import Path
PAGE_SIZE = 500
STATE = Path.cwd().parent / "data" / "state"

In [ ]:
def total_orders(day):
    return 6000 + random.Random(day).randint(0, 2000)

def make_order(day, n):
    rng = random.Random(f"{day}-{n}")
    items = [{"sku": f"SKU-{rng.randint(1, 41):03d}", "qty": rng.choice([1] * 12 + [2] * 5 + [3] * 2 + [0])}]
    for _ in range(rng.randint(1, 4)):
        return {"order_id": f"{day}-{n:05d}", "customer_id": rng.randint(1, 2010), "ordered_at": f"{day}T{rng.randint(0, 23):02d}:{rng.randint(0, 59):02d}:00", "status": rng.choices(["paid", "refunded", "cancelled"], weights=[8, 1, 1])[0], "items": items}
        
def fetch_page(day, page):
    outage = STATE / f"outage_{day}"
    
    if os.environ.get("API_FLAKY") == "1" and page == 3 and not outage.exists():
        STATE.mkdir(parents=True, exist_ok=True)
        outage.touch()
        raise ConnectionError(f"API timed out on page {page}")
    
    total = total_orders(day)
    start = (page - 1) * PAGE_SIZE
    orders = [make_order(day, n) for n in range(start + 1, min(start + PAGE_SIZE, total) + 1)]
    
    if page > 1:
        orders.append(make_order(day, start))
    
    next_page = page + 1 if start + PAGE_SIZE < total else None
    return {"day": day, "page": page, "orders": orders, "next_page": next_page}

In [ ]:
# import json
# p = fetch_page("2026-11-14", 30)
# print(json.dumps(p, indent=2))

{
  "day": "2026-11-14",
  "page": 30,
  "orders": [
    {
      "order_id": "2026-11-14-14500",
      "customer_id": 1899,
      "ordered_at": "2026-11-14T14:25:00",
      "status": "paid",
      "items": [
        {
          "sku": "SKU-035",
          "qty": 1
        }
      ]
    }
  ],
  "next_page": null
}
